In [2]:
#pip install medmnist

In [13]:
import torch
import numpy as np
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from medmnist import PneumoniaMNIST
from medmnist import INFO
from sklearn.metrics import roc_auc_score, classification_report

data_flag = 'pneumoniamnist'
info = INFO[data_flag]
DataClass = PneumoniaMNIST

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

In [5]:
train_dataset = DataClass(split='train', transform=transform, download=True)
val_dataset   = DataClass(split='val', transform=transform, download=True)
test_dataset  = DataClass(split='test', transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

  0%|          | 0/4170669 [00:00<?, ?it/s]

Using downloaded and verified file: C:\Users\CHARUSAT\.medmnist\pneumoniamnist.npz
Using downloaded and verified file: C:\Users\CHARUSAT\.medmnist\pneumoniamnist.npz
Train size: 4708
Val size: 524
Test size: 624


In [6]:
import torch.nn as nn

class MNISTCNN(nn.Module):
    def __init__(self):
        super(MNISTCNN, self).__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.AdaptiveAvgPool2d((1,1))
        )
        
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MNISTCNN().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [11]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        #labels = labels.float().unsqueeze(1).to(device)
        labels = labels.float().to(device)
        labels = labels.view(-1, 1)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.2666
Epoch 2, Loss: 0.1437
Epoch 3, Loss: 0.1084
Epoch 4, Loss: 0.0993
Epoch 5, Loss: 0.0839
Epoch 6, Loss: 0.0756
Epoch 7, Loss: 0.0708
Epoch 8, Loss: 0.0864
Epoch 9, Loss: 0.0542
Epoch 10, Loss: 0.0457
Epoch 11, Loss: 0.0446
Epoch 12, Loss: 0.0517
Epoch 13, Loss: 0.0371
Epoch 14, Loss: 0.0352
Epoch 15, Loss: 0.0310
Epoch 16, Loss: 0.0222
Epoch 17, Loss: 0.0119
Epoch 18, Loss: 0.0221
Epoch 19, Loss: 0.0205
Epoch 20, Loss: 0.0160


In [14]:
model.eval()
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs)
        
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

auc = roc_auc_score(all_labels, all_probs)
print("AUC:", auc)

AUC: 0.9654777558623713


In [15]:
from sklearn.metrics import classification_report, confusion_matrix

threshold = 0.5
y_pred = (all_probs > threshold).astype(int)

print(classification_report(all_labels, y_pred))
print(confusion_matrix(all_labels, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.87      0.88       234
           1       0.92      0.94      0.93       390

    accuracy                           0.91       624
   macro avg       0.91      0.91      0.91       624
weighted avg       0.91      0.91      0.91       624

[[204  30]
 [ 24 366]]


In [16]:
for t in [0.4, 0.5, 0.55, 0.6, 0.65]:
    y_pred = (all_probs > t).astype(int)
    print("Threshold:", t)
    print(classification_report(all_labels, y_pred))

Threshold: 0.4
              precision    recall  f1-score   support

           0       0.90      0.86      0.88       234
           1       0.92      0.94      0.93       390

    accuracy                           0.91       624
   macro avg       0.91      0.90      0.91       624
weighted avg       0.91      0.91      0.91       624

Threshold: 0.5
              precision    recall  f1-score   support

           0       0.89      0.87      0.88       234
           1       0.92      0.94      0.93       390

    accuracy                           0.91       624
   macro avg       0.91      0.91      0.91       624
weighted avg       0.91      0.91      0.91       624

Threshold: 0.55
              precision    recall  f1-score   support

           0       0.89      0.87      0.88       234
           1       0.92      0.94      0.93       390

    accuracy                           0.91       624
   macro avg       0.91      0.90      0.91       624
weighted avg       0.91     

In [17]:
from sklearn.metrics import accuracy_score

best_acc = 0
best_t = 0

for t in np.arange(0.3, 0.8, 0.01):
    y_pred = (all_probs > t).astype(int)
    acc = accuracy_score(all_labels, y_pred)
    
    if acc > best_acc:
        best_acc = acc
        best_t = t

print("Best Threshold:", best_t)
print("Best Accuracy:", best_acc)

Best Threshold: 0.4200000000000001
Best Accuracy: 0.9150641025641025


In [19]:
y_pred = (all_probs > best_t).astype(int)
cm = confusion_matrix(all_labels, y_pred)

print(cm)

[[203  31]
 [ 22 368]]


In [23]:
TN, FP, FN, TP = cm.ravel()
Sensitivity = TP / (TP + FN)
Specificity = TN / (TN + FP)
print(Sensitivity)
print(Specificity)

0.9435897435897436
0.8675213675213675
